In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("State_Wise_Preprocessing") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

spark


In [2]:
from pyspark.sql.functions import (
    col, trim, year, from_unixtime, to_date
)


In [3]:
df = spark.read.parquet(
    "hdfs://namenode:8020/google_local/processed/state=Alabama"
)

In [4]:
df.printSchema()

root
 |-- gmap_id: string (nullable = true)
 |-- rating: long (nullable = true)
 |-- text: string (nullable = true)
 |-- name: string (nullable = true)
 |-- category: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- avg_rating: double (nullable = true)
 |-- num_of_reviews: long (nullable = true)
 |-- state: string (nullable = true)
 |-- review_timestamp: string (nullable = true)
 |-- review_date: date (nullable = true)
 |-- year: integer (nullable = true)



In [ ]:
states = [
    "Alabama","Alaska","Arizona","Arkansas","Colorado",
    "Connecticut","Delaware","District_of_Columbia","Georgia","Hawaii","Idaho",
    "Indiana","Iowa","Kansas","Kentucky","Louisiana",
    "Maine","Maryland","Massachusetts","Michigan","Minnesota",
    "Mississippi","Missouri","Montana","Nebraska","Nevada",
    "New_Hampshire","New_Jersey","New_Mexico",
    "North_Carolina","North_Dakota","Ohio","Oklahoma","Oregon",
    "Rhode_Island","South_Carolina","South_Dakota",
    "Tennessee","Utah","Vermont","Virginia","Washington",
    "West_Virginia","Wisconsin","Wyoming"
]

In [ ]:
for state in states:
    print(f"Processing state: {state}")

    # 1. Read reviews
    reviews_df = spark.read.json(
        f"hdfs://namenode:8020/google_local/total_reviews_raw/state={state}"
    )

    # 2. Read metadata
    metadata_df = spark.read.json(
        f"hdfs://namenode:8020/google_local/total_metadata_raw/state={state}"
    )

    # 3. Select required columns
    reviews_clean = reviews_df.select(
        "gmap_id", "rating", "text", "time"
    )

    metadata_clean = metadata_df.select(
        "gmap_id",
        "name",
        "category",
        "avg_rating",
        "num_of_reviews",
        "state"
    )

    # 4. Join
    joined_df = reviews_clean.join(
        metadata_clean,
        on="gmap_id",
        how="inner"
    )

    # 5. Timestamp conversion
    joined_df = joined_df.withColumn(
        "review_timestamp",
        from_unixtime(col("time") / 1000)
    ).withColumn(
        "review_date",
        to_date(col("review_timestamp"))
    ).withColumn(
        "year",
        year(col("review_date"))
    )

    # 6. Drop raw time
    joined_df = joined_df.drop("time")

    # 7. Text + rating cleaning
    joined_df = joined_df.filter(
        (col("text").isNotNull()) &
        (trim(col("text")) != "") &
        (col("rating").between(1, 5)) &
        (col("year").isNotNull())
    )

    # 8. Write (ONE WRITE ONLY)
    (
        joined_df
        .write
        .mode("overwrite")
        .partitionBy("year")
        .parquet(
            f"hdfs://namenode:8020/google_local/processed/state={state}"
        )
    )

    # 9. Clear memory
    spark.catalog.clearCache()

    print(f"Finished state: {state}")


In [ ]:
states1 = [
    "Pennsylvania","Illinois"
]

In [ ]:
for state in states1:
    print(f"Processing state: {state}")

    # 1. Read reviews
    reviews_df = spark.read.json(
        f"hdfs://namenode:8020/google_local/total_reviews_raw/state={state}"
    )

    # 2. Read metadata
    metadata_df = spark.read.json(
        f"hdfs://namenode:8020/google_local/total_metadata_raw/state={state}"
    )

    # 3. Select required columns
    reviews_clean = reviews_df.select(
        "gmap_id", "rating", "text", "time"
    )

    metadata_clean = metadata_df.select(
        "gmap_id",
        "name",
        "category",
        "avg_rating",
        "num_of_reviews",
        "state"
    )

    # 4. Join
    joined_df = reviews_clean.join(
        metadata_clean,
        on="gmap_id",
        how="inner"
    )

    # 5. Timestamp conversion
    joined_df = joined_df.withColumn(
        "review_timestamp",
        from_unixtime(col("time") / 1000)
    ).withColumn(
        "review_date",
        to_date(col("review_timestamp"))
    ).withColumn(
        "year",
        year(col("review_date"))
    )

    # 6. Drop raw time
    joined_df = joined_df.drop("time")

    # 7. Text + rating cleaning
    joined_df = joined_df.filter(
        (col("text").isNotNull()) &
        (trim(col("text")) != "") &
        (col("rating").between(1, 5)) &
        (col("year").isNotNull())
    )

    # 8. Write (ONE WRITE ONLY)
    (
        joined_df
        .write
        .mode("overwrite")
        .partitionBy("year")
        .parquet(
            f"hdfs://namenode:8020/google_local/processed/state={state}"
        )
    )

    # 9. Clear memory
    spark.catalog.clearCache()

    print(f"Finished state: {state}")


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Preprocess_Big_State") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .getOrCreate()

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Preprocessing_California") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020") \
    .getOrCreate()


In [ ]:
reviews_df = spark.read.json(
    "hdfs://namenode:8020/google_local/total_reviews_raw/state=California"
)

metadata_df = spark.read.json(
    "hdfs://namenode:8020/google_local/total_metadata_raw/state=California"
)

reviews_df = reviews_df.select(
    "gmap_id", "rating", "text", "time"
)

metadata_df = metadata_df.select(
    "gmap_id", "name", "category",
    "avg_rating", "num_of_reviews", "state"
)

joined_df = reviews_df.join(
    metadata_df,
    on="gmap_id",
    how="inner"
)

In [ ]:
from pyspark.sql.functions import from_unixtime, year, col, to_date

joined_df = joined_df.withColumn(
        "review_timestamp",
        from_unixtime(col("time") / 1000)
    ).withColumn(
        "review_date",
        to_date(col("review_timestamp"))
    ).withColumn(
        "year",
        year(col("review_date"))
    )

In [ ]:
from pyspark.sql.functions import trim

joined_df = joined_df.filter(
    (col("text").isNotNull()) &
    (trim(col("text")) != "") &
    (col("rating").between(1, 5)) &
    (col("year").isNotNull())
).drop("time")


In [ ]:
joined_df.printSchema()

In [ ]:
from pyspark.sql.functions import year

years = (
    joined_df
    .select(year("review_date").alias("y"))
    .distinct()
    .orderBy("y")
    .collect()
)

years = [row.y for row in years if row.y is not None]
years

In [ ]:
from pyspark.sql.functions import year

for y in years:
    print(f"Writing year {y}...")
    
    (
        joined_df
        .filter(year("review_date") == y)
        .coalesce(2)
        .write
        .mode("overwrite")
        .parquet(f"hdfs:///google_local/processed/state=California/year={y}")
    )

In [ ]:
spark.catalog.clearCache()
spark.stop()

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Preprocessing_Florida") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020") \
    .getOrCreate()

In [ ]:
reviews_df = spark.read.json(
    "hdfs://namenode:8020/google_local/total_reviews_raw/state=Florida"
)

metadata_df = spark.read.json(
    "hdfs://namenode:8020/google_local/total_metadata_raw/state=Florida"
)

reviews_df = reviews_df.select(
    "gmap_id", "rating", "text", "time"
)

metadata_df = metadata_df.select(
    "gmap_id", "name", "category",
    "avg_rating", "num_of_reviews", "state"
)

joined_df = reviews_df.join(
    metadata_df,
    on="gmap_id",
    how="inner"
)

In [ ]:
from pyspark.sql.functions import from_unixtime, year, col, to_date

joined_df = joined_df.withColumn(
        "review_timestamp",
        from_unixtime(col("time") / 1000)
    ).withColumn(
        "review_date",
        to_date(col("review_timestamp"))
    ).withColumn(
        "year",
        year(col("review_date"))
    )

In [ ]:
from pyspark.sql.functions import trim

joined_df = joined_df.filter(
    (col("text").isNotNull()) &
    (trim(col("text")) != "") &
    (col("rating").between(1, 5)) &
    (col("year").isNotNull())
).drop("time")


In [ ]:
joined_df.printSchema()

In [ ]:
from pyspark.sql.functions import year

years = (
    joined_df
    .select(year("review_date").alias("y"))
    .distinct()
    .orderBy("y")
    .collect()
)

years = [row.y for row in years if row.y is not None]
years

In [ ]:
from pyspark.sql.functions import year

for y in years:
    print(f"Writing year {y}...")
    
    (
        joined_df
        .filter(year("review_date") == y)
        .coalesce(2)
        .write
        .mode("overwrite")
        .parquet(f"hdfs:///google_local/processed/state=Florida/year={y}")
    )

In [ ]:
spark.catalog.clearCache()
spark.stop()

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Preprocessing_Florida") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020") \
    .getOrCreate()

In [ ]:
reviews_df = spark.read.json(
    "hdfs://namenode:8020/google_local/total_reviews_raw/state=Texas"
)

metadata_df = spark.read.json(
    "hdfs://namenode:8020/google_local/total_metadata_raw/state=Texas"
)

reviews_df = reviews_df.select(
    "gmap_id", "rating", "text", "time"
)

metadata_df = metadata_df.select(
    "gmap_id", "name", "category",
    "avg_rating", "num_of_reviews", "state"
)

joined_df = reviews_df.join(
    metadata_df,
    on="gmap_id",
    how="inner"
)

In [ ]:
from pyspark.sql.functions import from_unixtime, year, col, to_date

joined_df = joined_df.withColumn(
        "review_timestamp",
        from_unixtime(col("time") / 1000)
    ).withColumn(
        "review_date",
        to_date(col("review_timestamp"))
    ).withColumn(
        "year",
        year(col("review_date"))
    )

In [ ]:
from pyspark.sql.functions import trim

joined_df = joined_df.filter(
    (col("text").isNotNull()) &
    (trim(col("text")) != "") &
    (col("rating").between(1, 5)) &
    (col("year").isNotNull())
).drop("time")


In [ ]:
joined_df.printSchema()

In [ ]:
from pyspark.sql.functions import year

years = (
    joined_df
    .select(year("review_date").alias("y"))
    .distinct()
    .orderBy("y")
    .collect()
)

years = [row.y for row in years if row.y is not None]
years

In [ ]:
from pyspark.sql.functions import year

for y in years:
    print(f"Writing year {y}...")
    
    (
        joined_df
        .filter(year("review_date") == y)
        .coalesce(2)
        .write
        .mode("overwrite")
        .parquet(f"hdfs:///google_local/processed/state=Texas/year={y}")
    )

In [ ]:
spark.catalog.clearCache()
spark.stop()

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Preprocessing_New_York") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:8020") \
    .getOrCreate()

In [ ]:
reviews_df = spark.read.json(
    "hdfs://namenode:8020/google_local/total_reviews_raw/state=New_York"
)

metadata_df = spark.read.json(
    "hdfs://namenode:8020/google_local/total_metadata_raw/state=New_York"
)

reviews_df = reviews_df.select(
    "gmap_id", "rating", "text", "time"
)

metadata_df = metadata_df.select(
    "gmap_id", "name", "category",
    "avg_rating", "num_of_reviews", "state"
)

joined_df = reviews_df.join(
    metadata_df,
    on="gmap_id",
    how="inner"
)

In [ ]:
from pyspark.sql.functions import from_unixtime, year, col, to_date

joined_df = joined_df.withColumn(
        "review_timestamp",
        from_unixtime(col("time") / 1000)
    ).withColumn(
        "review_date",
        to_date(col("review_timestamp"))
    ).withColumn(
        "year",
        year(col("review_date"))
    )

In [ ]:
from pyspark.sql.functions import trim

joined_df = joined_df.filter(
    (col("text").isNotNull()) &
    (trim(col("text")) != "") &
    (col("rating").between(1, 5)) &
    (col("year").isNotNull())
).drop("time")


In [ ]:
joined_df.printSchema()

In [ ]:
from pyspark.sql.functions import year

years = (
    joined_df
    .select(year("review_date").alias("y"))
    .distinct()
    .orderBy("y")
    .collect()
)

years = [row.y for row in years if row.y is not None]
years

In [ ]:
from pyspark.sql.functions import year

for y in years:
    print(f"Writing year {y}...")
    
    (
        joined_df
        .filter(year("review_date") == y)
        .coalesce(2)
        .write
        .mode("overwrite")
        .parquet(f"hdfs:///google_local/processed/state=New_York/year={y}")
    )